In [1]:
import pandas as pd
import numpy as np
from pathlib import Path


In [2]:
BASE_PATH = Path.cwd().parents[0]

BRONZE_PATH = BASE_PATH / "data" / "bronze"
SILVER_PATH = BASE_PATH / "data" / "silver"

SILVER_PATH.mkdir(parents=True, exist_ok=True)


In [3]:
import pandas as pd

# ---------------------------------------
# AIS (LA scoped, point-level)
# ---------------------------------------
ais_df = pd.read_csv(
    BRONZE_PATH / "final_combined_df.csv",
    parse_dates=["BaseDateTime"]
)

# ---------------------------------------
# Port Calls (event-level)
# ---------------------------------------
port_call_df = pd.read_csv(
    BRONZE_PATH / "la_port_calls_2024.csv",
    parse_dates=["ArrivalTime", "BerthStartTime", "DepartureTime"]
)

# ---------------------------------------
# Port Operations (capacity-level)
# ---------------------------------------
port_ops_df = pd.read_csv(
    BRONZE_PATH / "la_port_oprs_2024.csv"
)

# Ensure Date column exists and is correct type
port_ops_df["Date"] = pd.to_datetime(port_ops_df["Date"]).dt.date


In [4]:
ais_df.shape, port_call_df.shape, port_ops_df.shape

((4177218, 18), (1112, 12), (25620, 10))

In [5]:
ais_df["MMSI"].value_counts()

MMSI
368237190    135365
366734000    133350
368270970    131871
368850000     79979
366791000     23575
              ...  
566886000        10
219130000         7
413173000         5
563181600         3
636093279         2
Name: count, Length: 1293, dtype: int64

In [6]:
ais_features = (
    ais_df
    .groupby("MMSI")
    .agg(
        AvgSOG=("SOG", "mean"),
        MaxSOG=("SOG", "max"),
        AvgCOG=("COG", "mean"),
        AISPoints=("SOG", "count"),
        WaitingPoints=("SOG", lambda x: (x <= 0.5).sum()),
        VesselDraft=("Draft", "max"),
        VesselLength=("Length", "max"),
        VesselWidth=("Width", "max"),
    )
    .reset_index()
)

ais_features.head()


,MMSI,AvgSOG,MaxSOG,AvgCOG,AISPoints,WaitingPoints,VesselDraft,VesselLength,VesselWidth
0,205717000,0.051193,0.5,174.896964,5237,5237,6.7,199.0,32.0
1,209652000,0.001494,0.5,119.284952,937,937,10.9,229.0,32.0
2,209889000,0.001242,0.4,110.716948,1127,1127,11.9,229.0,36.0
3,210136000,0.005674,0.5,154.477935,2538,2538,8.0,229.0,36.0
4,210185000,0.143333,0.5,221.173333,90,90,10.5,189.0,28.0


In [7]:
pc_ais_df = port_call_df.merge(
    ais_features,
    on="MMSI",
    how="left"
)


In [8]:
pc_ais_df.shape


(1112, 20)

In [9]:
# Ensure safe assignment (avoid SettingWithCopyWarning)
pc_ais_df = pc_ais_df.copy()
port_ops_df = port_ops_df.copy()

# Align Date columns for join
pc_ais_df["Date"] = pd.to_datetime(pc_ais_df["ArrivalTime"]).dt.date
port_ops_df["Date"] = pd.to_datetime(port_ops_df["Date"]).dt.date


In [10]:
# Define berth time slots
time_slots = [
    ("00:00", "08:00"),
    ("08:00", "16:00"),
    ("16:00", "24:00")
]

# Expand port_ops_df by time slots
port_ops_expanded = []

for _, row in port_ops_df.iterrows():
    for slot_start, slot_end in time_slots:
        new_row = row.copy()
        new_row["SlotStart"] = slot_start
        new_row["SlotEnd"] = slot_end
        port_ops_expanded.append(new_row)

port_ops_df = pd.DataFrame(port_ops_expanded)

print("Expanded Port Ops:", port_ops_df.shape)


Expanded Port Ops: (76860, 12)


In [11]:
# Merge Port Calls + AIS features with Port Operations
final_df = pc_ais_df.merge(
    port_ops_df,
    on=["PortCode", "Date"],
    how="left",
    validate="many_to_many"
)


In [12]:
final_df["ArrivalHour"] = pd.to_datetime(final_df["ArrivalTime"]).dt.hour
final_df["ArrivalDayOfWeek"] = pd.to_datetime(final_df["ArrivalTime"]).dt.dayofweek
final_df["IsWeekend"] = final_df["ArrivalDayOfWeek"].isin([5, 6]).astype(int)

In [13]:
final_df["CongestionLevel"] = pd.cut(
    final_df["AnchorageWaitingTime"],
    bins=[-1, 60, np.inf],
    labels=["Low", "High"]
)

In [14]:
final_df = final_df.rename(columns={
    "VesselDraft": "VesselDraft",        # keep as-is (explicit for clarity)
    "BerthMaxDraft": "BerthMaxDraft"     # keep as-is
})

In [15]:
final_df["BerthFeasible"] = (
    (final_df["VesselLength"] <= final_df["MaxLOA"]) &
    (final_df["VesselDraft"] <= final_df["BerthDepth"])
).astype(int)

In [16]:
final_df.shape

(233520, 36)

In [17]:
final_df.head()

,PortCallID,MMSI,IMO,VesselName,PortCode,ArrivalTime,BerthStartTime,DepartureTime,AnchorageWaitingTime,BerthTime,...,BerthMaxDraft,DailyCapacityTEU,OperationalStatus,SlotStart,SlotEnd,ArrivalHour,ArrivalDayOfWeek,IsWeekend,CongestionLevel,BerthFeasible
0,e3e10dc8-bd0d-4662-a61a-dfbcb74651ff,205717000,IMO9748485,LA TONDA,USLAX,2024-04-21 13:32:10,2024-04-21 13:32:10,2024-05-01 06:14:59,21083.23,232.71,...,14.9,15298,Maintenance,00:00,08:00,13,6,1,High,1
1,e3e10dc8-bd0d-4662-a61a-dfbcb74651ff,205717000,IMO9748485,LA TONDA,USLAX,2024-04-21 13:32:10,2024-04-21 13:32:10,2024-05-01 06:14:59,21083.23,232.71,...,14.9,15298,Maintenance,08:00,16:00,13,6,1,High,1
2,e3e10dc8-bd0d-4662-a61a-dfbcb74651ff,205717000,IMO9748485,LA TONDA,USLAX,2024-04-21 13:32:10,2024-04-21 13:32:10,2024-05-01 06:14:59,21083.23,232.71,...,14.9,15298,Maintenance,16:00,24:00,13,6,1,High,1
3,e3e10dc8-bd0d-4662-a61a-dfbcb74651ff,205717000,IMO9748485,LA TONDA,USLAX,2024-04-21 13:32:10,2024-04-21 13:32:10,2024-05-01 06:14:59,21083.23,232.71,...,14.5,18334,Normal,00:00,08:00,13,6,1,High,1
4,e3e10dc8-bd0d-4662-a61a-dfbcb74651ff,205717000,IMO9748485,LA TONDA,USLAX,2024-04-21 13:32:10,2024-04-21 13:32:10,2024-05-01 06:14:59,21083.23,232.71,...,14.5,18334,Normal,08:00,16:00,13,6,1,High,1


In [18]:
final_df.to_csv(
    SILVER_PATH / "la_integrated_dataset_2024.csv",
    index=False
)


In [19]:
final_df.columns

Index(['PortCallID', 'MMSI', 'IMO', 'VesselName', 'PortCode', 'ArrivalTime',
       'BerthStartTime', 'DepartureTime', 'AnchorageWaitingTime', 'BerthTime',
       'HandledTEU', 'PortCallStatus', 'AvgSOG', 'MaxSOG', 'AvgCOG',
       'AISPoints', 'WaitingPoints', 'VesselDraft', 'VesselLength',
       'VesselWidth', 'Date', 'TerminalID', 'BerthID', 'BerthLength',
       'BerthDepth', 'MaxLOA', 'BerthMaxDraft', 'DailyCapacityTEU',
       'OperationalStatus', 'SlotStart', 'SlotEnd', 'ArrivalHour',
       'ArrivalDayOfWeek', 'IsWeekend', 'CongestionLevel', 'BerthFeasible'],
      dtype='object')

In [20]:
final_df["BerthID"].value_counts()

BerthID
B1     23352
B2     23352
B3     23352
B4     23352
B5     23352
B6     23352
B7     23352
B8     23352
B9     23352
B10    23352
Name: count, dtype: int64

In [21]:
final_df['TerminalID'].value_counts()

TerminalID
T1    33360
T2    33360
T3    33360
T4    33360
T5    33360
T6    33360
T7    33360
Name: count, dtype: int64